# Prophet Evaluation — Regime-Based Benchmark

Mirrors the TimesFM evaluation in notebook 06 but uses **Facebook Prophet** with UK public holidays and tuned seasonality parameters. Evaluated at **force level** (all 16 crime types aggregated) across the same 3 train/test regimes.

| Regime | Train | Test |
|---|---|---|
| Pre-pandemic | 2012–2018 | 2019 |
| Post-pandemic | 2022–2023 | 2024 |
| Combined non-pandemic | 2012–2019 + 2022–2024 | 2025 |

**Baseline:** per-force mean monthly count in training window  
**Forces:** Cheshire, Lincolnshire, Merseyside, Metropolitan, West Midlands

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display

## 1. Load and aggregate to force level

In [ ]:
FORCES = {
    'Cheshire Constabulary':         'Cheshire',
    'Lincolnshire Police':            'Lincolnshire',
    'Merseyside Police':              'Merseyside',
    'Metropolitan Police Service':    'Metropolitan',
    'West Midlands Police':           'West Midlands',
}
CRIME_TYPES = [
    'Violence and sexual offences', 'Criminal damage and arson', 'Drugs',
    'Violent crime', 'Burglary', 'Other theft', 'Vehicle crime',
    'Public order', 'Other crime', 'Shoplifting', 'Robbery',
    'Bicycle theft', 'Theft from the person', 'Possession of weapons',
    'Public disorder and weapons', 'Anti-social behaviour',
]
REGIMES = {
    'Pre-pandemic': {
        'train_years': set(range(2012, 2019)),
        'test_years':  {2019},
    },
    'Post-pandemic': {
        'train_years': {2022, 2023},
        'test_years':  {2024},
    },
    'Combined non-pandemic': {
        'train_years': set(range(2012, 2020)) | {2022, 2023, 2024},
        'test_years':  {2025},
    },
}
REGIME_ORDER = ['Pre-pandemic', 'Post-pandemic', 'Combined non-pandemic']

raw = pd.read_parquet('../data/processed/crimes_clean_dedup_all_years.parquet')
raw = raw[raw['Falls within'].isin(FORCES) & raw['Crime type'].isin(CRIME_TYPES)].copy()
raw['force'] = raw['Falls within'].map(FORCES)
raw['month'] = pd.to_datetime(raw['Month'])
raw = raw[raw['month'].dt.year >= 2012]

force_monthly = (
    raw.groupby(['force', 'month'])
    .size()
    .reset_index(name='count')
)
force_labels = list(FORCES.values())
print(f'Date range: {force_monthly["month"].min().date()} → {force_monthly["month"].max().date()}')
print(f'Forces: {force_labels}')

## 2. Prophet model builder

Key parameters:
- `seasonality_mode='multiplicative'` — crime demand scales with level, not additive
- `yearly_seasonality=True` — strong annual patterns (summer peaks etc.)
- `weekly_seasonality=False` — monthly data, no weekly signal
- `daily_seasonality=False` — monthly data
- UK public holidays added via `add_country_holidays('GB')`
- `changepoint_prior_scale=0.15` — moderate flexibility for trend changes
- `seasonality_prior_scale=10` — allow strong seasonality

In [ ]:
# metric helpers (defined once, reused across the notebook)
def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))

def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

def mape(pred, actual):
    # guard against division by zero where actual demand is 0
    return float(np.mean(np.abs((pred - actual) / np.where(actual == 0, 1, actual))) * 100)

def build_prophet():
    """Build a Prophet model with the parameters used across all forces."""
    m = Prophet(
        seasonality_mode='multiplicative',
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.15,
        seasonality_prior_scale=10,
        uncertainty_samples=0,  # point forecasts only, so skip sampling for speed
    )
    m.add_country_holidays(country_name='GB')
    return m


def prophet_forecast(train_series, n_months):
    """Fit Prophet on a monthly series and return n_months point forecasts."""
    df = train_series.reset_index()
    df.columns = ['ds', 'y']
    df['ds'] = pd.to_datetime(df['ds'])

    m = build_prophet()
    m.fit(df)

    future = m.make_future_dataframe(periods=n_months, freq='MS', include_history=False)
    forecast = m.predict(future)
    return np.clip(forecast['yhat'].values[:n_months], 0, None)

## 3. Run evaluation across all regimes

In [ ]:
results = []
total = len(REGIME_ORDER) * len(force_labels)
done = 0

for regime_name in REGIME_ORDER:
    cfg = REGIMES[regime_name]
    for force in force_labels:
        done += 1
        print(f'[{done:02d}/{total}] {regime_name} | {force}', end='  ')

        series = force_monthly[
            (force_monthly['force'] == force)
        ].set_index('month')['count']

        train = series[series.index.year.isin(cfg['train_years'])].sort_index()
        test  = series[series.index.year.isin(cfg['test_years'])].sort_index()

        if train.empty or test.empty:
            print('SKIPPED')
            continue

        n_test = len(test)
        baseline_pred = np.full(n_test, train.mean())
        prophet_pred  = prophet_forecast(train, n_test)

        actual = test.values.astype(float)
        results.append({
            'regime': regime_name,
            'force': force,
            'n_test': n_test,
            'baseline_mae':   mae(baseline_pred, actual),
            'model_mae':      mae(prophet_pred,  actual),
            'baseline_rmse':  rmse(baseline_pred, actual),
            'model_rmse':     rmse(prophet_pred,  actual),
            'model_mape':     mape(prophet_pred,  actual),
        })

        print(f"baseline={results[-1]['baseline_mae']:.1f}  prophet={results[-1]['model_mae']:.1f}  MAPE={results[-1]['model_mape']:.1f}%")

results_df = pd.DataFrame(results)
results_df['delta_mae']   = results_df['baseline_mae'] - results_df['model_mae']
results_df['rmae']        = results_df['model_mae'] / results_df['baseline_mae']
results_df['model_wins']  = results_df['model_mae'] < results_df['baseline_mae']
print(f'\nDone. {len(results_df)} combos evaluated.')

## 4. Headline results by regime

In [ ]:
headline = (
    results_df.groupby('regime')
    .agg(
        baseline_mae  = ('baseline_mae',  'mean'),
        model_mae     = ('model_mae',     'mean'),
        baseline_rmse = ('baseline_rmse', 'mean'),
        model_rmse    = ('model_rmse',    'mean'),
        delta_mae     = ('delta_mae',     'mean'),
        rmae          = ('rmae',          'mean'),
        model_mape    = ('model_mape',    'mean'),
        win_rate      = ('model_wins',    'mean'),
    )
    .reset_index()
)
headline['win_pct'] = (headline['win_rate'] * 100).round(0).astype(int)
headline['regime']  = pd.Categorical(headline['regime'], categories=REGIME_ORDER, ordered=True)
headline = headline.sort_values('regime').reset_index(drop=True)

display(
    headline[['regime','baseline_mae','model_mae','delta_mae','rmae','model_mape','baseline_rmse','model_rmse','win_pct']]
    .round(2)
    .rename(columns={
        'regime':'Regime','baseline_mae':'Baseline MAE','model_mae':'Prophet MAE',
        'delta_mae':'Δ MAE','rmae':'RMAE','model_mape':'MAPE %',
        'baseline_rmse':'Baseline RMSE','model_rmse':'Prophet RMSE','win_pct':'Win%'
    })
    .set_index('Regime')
)

## 5. Results by force

In [ ]:
by_force = (
    results_df.groupby(['regime', 'force'])
    .agg(delta_mae=('delta_mae','mean'), rmae=('rmae','mean'),
         model_mape=('model_mape','mean'), win_rate=('model_wins','mean'))
    .reset_index()
)
by_force['win_pct'] = (by_force['win_rate'] * 100).round(0).astype(int)

delta_pivot = by_force.pivot_table(index='force', columns='regime', values='delta_mae')
delta_pivot = delta_pivot[[c for c in REGIME_ORDER if c in delta_pivot.columns]]

mape_pivot = by_force.pivot_table(index='force', columns='regime', values='model_mape')
mape_pivot = mape_pivot[[c for c in REGIME_ORDER if c in mape_pivot.columns]]

print('=== Δ MAE by force (positive = Prophet beats baseline) ===')
display(delta_pivot.round(2))
print('\n=== MAPE % by force ===')
display(mape_pivot.round(1))

## 6. TimesFM vs Prophet vs Baseline comparison

In [ ]:
# TimesFM force-level headline numbers from notebook 06
# (these were LSOA-level and different crime subset — noted for context)
timesfm_headline = pd.DataFrame([
    {'regime': 'Pre-pandemic',          'tfm_mae': 0.509, 'tfm_rmae': 0.837, 'tfm_win_pct': 96},
    {'regime': 'Post-pandemic',         'tfm_mae': 0.472, 'tfm_rmae': 0.810, 'tfm_win_pct': 100},
    {'regime': 'Combined non-pandemic', 'tfm_mae': 0.458, 'tfm_rmae': 0.720, 'tfm_win_pct': 100},
])

comp = headline[['regime','baseline_mae','model_mae','rmae','win_pct']].rename(
    columns={'model_mae':'prophet_mae','rmae':'prophet_rmae','win_pct':'prophet_win_pct'}
).merge(timesfm_headline, on='regime')

comp['regime'] = pd.Categorical(comp['regime'], categories=REGIME_ORDER, ordered=True)
comp = comp.sort_values('regime').reset_index(drop=True)

print('Note: TimesFM numbers are LSOA-level (5 crime types); Prophet numbers are force-level (16 crime types)')
print('RMAE < 1.0 = beats baseline; lower is better\n')
display(
    comp[['regime','baseline_mae','prophet_mae','prophet_rmae','prophet_win_pct','tfm_mae','tfm_rmae','tfm_win_pct']]
    .round(3)
    .rename(columns={
        'regime':'Regime','baseline_mae':'Baseline MAE',
        'prophet_mae':'Prophet MAE','prophet_rmae':'Prophet RMAE','prophet_win_pct':'Prophet Win%',
        'tfm_mae':'TimesFM MAE','tfm_rmae':'TimesFM RMAE','tfm_win_pct':'TFM Win%',
    })
    .set_index('Regime')
)

## 7. Visualisations

In [ ]:
# Bar chart: Baseline vs Prophet MAE by regime
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Prophet vs Baseline MAE by Regime', fontsize=13, fontweight='bold')

for ax, regime in zip(axes, REGIME_ORDER):
    row = headline[headline['regime'] == regime].iloc[0]
    vals   = [row['baseline_mae'], row['model_mae']]
    labels = ['Baseline', 'Prophet']
    colors = ['#9E9E9E', '#4CAF50']
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=0.8, width=0.5)
    ax.set_title(regime, fontsize=10, fontweight='bold')
    ax.set_ylabel('Mean MAE')
    ax.set_ylim(0, max(vals) * 1.25)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.02,
                f'{v:.1f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/prophet_mae_by_regime.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Δ MAE by force × regime
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    delta_pivot.round(1), annot=True, fmt='.1f', cmap='RdYlGn',
    center=0, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Δ MAE (positive = Prophet better than baseline)'}
)
ax.set_title('Prophet: Δ MAE by Force × Regime', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../outputs/prophet_delta_mae_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Forecast vs actual plots for each force × regime
fig, axes = plt.subplots(3, 5, figsize=(20, 12), sharey=False)
fig.suptitle('Prophet Forecast vs Actual by Regime × Force', fontsize=13, fontweight='bold')

for row_i, regime_name in enumerate(REGIME_ORDER):
    cfg = REGIMES[regime_name]
    for col_i, force in enumerate(force_labels):
        ax = axes[row_i][col_i]

        series = force_monthly[force_monthly['force'] == force].set_index('month')['count']
        train  = series[series.index.year.isin(cfg['train_years'])].sort_index()
        test   = series[series.index.year.isin(cfg['test_years'])].sort_index()

        prophet_pred = prophet_forecast(train, len(test))
        baseline_val = train.mean()

        # Recent training history for context
        ax.plot(train.index[-24:], train.values[-24:], color='#94a3b8', linewidth=1, label='History')
        ax.plot(test.index, test.values, color='#FF5722', linewidth=2, marker='o', markersize=4, label='Actual')
        ax.plot(test.index, prophet_pred, color='#4CAF50', linewidth=2, marker='s', markersize=4, linestyle='--', label='Prophet')
        ax.axhline(baseline_val, color='#9E9E9E', linestyle=':', linewidth=1, label='Baseline')

        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b%y'))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        ax.tick_params(axis='x', labelsize=6, rotation=30)
        ax.tick_params(axis='y', labelsize=7)

        if row_i == 0:
            ax.set_title(force, fontsize=9, fontweight='bold')
        if col_i == 0:
            ax.set_ylabel(regime_name.replace(' ', '\n'), fontsize=8, fontweight='bold')

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig('../outputs/prophet_forecast_vs_actual_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save results

In [ ]:
results_df.to_csv('../outputs/prophet_evaluation_results.csv', index=False)
comp.to_csv('../outputs/prophet_vs_timesfm_comparison.csv', index=False)
print('Saved to outputs/')